# Connect to BG Mono DB

In [ ]:
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()

# Get database credentials from environment variables
db_user = os.getenv("db_user")
db_password = os.getenv("db_password")
db_host = os.getenv("db_host")
db_port = os.getenv("db_port")
db_name = os.getenv("db_name")

print(f"Connecting to: {db_host}:{db_port}/{db_name}")

'db.dkyfdmfgnfxfwfhhxxvn.supabase.co'

In [ ]:

# Create SQLAlchemy engine (better than raw psycopg2)
# SQLAlchemy provides better connection pooling, error handling, and pandas integration
try:
    # Build connection string
    connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
    
    # Create engine with SSL support for Supabase
    engine = create_engine(
        connection_string,
        connect_args={"sslmode": "require"}  # Required for Supabase
    )
    
    # Test the connection
    with engine.connect() as connection:
        result = connection.execute(text("SELECT NOW();"))
        current_time = result.fetchone()[0]
        print("✅ Connection successful!")
        print(f"Current Time: {current_time}")
    
    print("✅ Database engine created successfully!")
    
except Exception as e:
    print(f"❌ Failed to connect: {e}")
    print("Make sure your .env file has the correct database credentials")

ModuleNotFoundError: No module named 'psycopg2'

In [ ]:
# Query data using pandas with SQLAlchemy engine (no more warnings!)
query = """SELECT * FROM security_master.security LIMIT 10"""
result = pd.read_sql(query, engine)  # Using engine instead of raw connection
print(f"Found {len(result)} records")
result

C:\Users\User\AppData\Local\Temp\ipykernel_16576\2456437720.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, connection)


,security_id,company_id,name,security_type,cusip,isin,share_class_figi,underlying_security_id


In [ ]:
# Additional SQLAlchemy features and best practices

# 1. Execute raw SQL queries
def execute_query(sql_query):
    """Execute a SQL query and return results as DataFrame"""
    try:
        return pd.read_sql(sql_query, engine)
    except Exception as e:
        print(f"Query failed: {e}")
        return None

# 2. Get table information
tables_query = """
SELECT table_schema, table_name 
FROM information_schema.tables 
WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
ORDER BY table_schema, table_name;
"""

print("📊 Available tables:")
tables = execute_query(tables_query)
if tables is not None:
    print(tables)

# 3. Get security_master schema info
schema_info = execute_query("""
SELECT column_name, data_type, is_nullable
FROM information_schema.columns 
WHERE table_schema = 'security_master' 
AND table_name = 'security'
ORDER BY ordinal_position;
""")

if schema_info is not None:
    print("\n🔍 Security table schema:")
    print(schema_info)
